# Universal Data Profiling Template
**UC20 — Enterprise Financial Data Platform | Sprint 2**

Use this notebook to profile any PostgreSQL staging table. Change `TABLE_NAME` for each assigned table. It reports data-quality patterns without assuming a business key, deleting duplicates, filling missing values, or modifying PostgreSQL.

**Safety:** this notebook runs read-only `SELECT` queries.

## 1. Imports and database connection

In [47]:
import getpass
import re
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

DB_USER = "postgres"
DB_PASSWORD = getpass.getpass("PostgreSQL password: ")
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "data_profiling"

connection_url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(connection_url)

with engine.connect() as conn:
    print("Connected to database:", conn.execute(text("SELECT current_database()")).scalar())

Connected to database: data_profiling


## 2. Choose the table

Change `TABLE_NAME` for each table. Examples might include `stg_ska1`, `stg_skat`, `stg_lfa1`, or `stg_t001`; use the actual names in your database.

In [48]:
SCHEMA_NAME = "public"
TABLE_NAME = "stg_bseg"  # <-- CHANGE THIS for each assigned table

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA_NAME):
    raise ValueError("Invalid schema identifier")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", TABLE_NAME):
    raise ValueError("Invalid table identifier")

qualified_table = f'"{SCHEMA_NAME}"."{TABLE_NAME}"'

## 3. Confirm the table exists and inspect its schema

In [49]:
schema_query = text("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY ordinal_position
""")

with engine.connect() as conn:
    schema_df = pd.read_sql(
        schema_query, conn,
        params={"schema_name": SCHEMA_NAME, "table_name": TABLE_NAME}
    )

if schema_df.empty:
    raise ValueError(f"Table {SCHEMA_NAME}.{TABLE_NAME} was not found or has no visible columns.")

display(schema_df)

,column_name,data_type,is_nullable
0,client_id,character varying,YES
1,company_code,character varying,YES
2,accounting_document_no,character varying,YES
3,fiscal_year,integer,YES
4,line_item_no,character varying,YES
5,account_type,character varying,YES
6,debit_credit_indicator,character varying,YES
7,amount_local_currency,numeric,YES
8,amount_document_currency,numeric,YES
9,assignment_no,character varying,YES


## 4. Load table and inspect basic structure

In [50]:
df = pd.read_sql(f"SELECT * FROM {qualified_table}", engine)

print(f"Table: {SCHEMA_NAME}.{TABLE_NAME}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())
display(df.dtypes.rename("pandas_dtype").to_frame())

Table: public.stg_bseg
Rows: 332,106
Columns: 21


,client_id,company_code,accounting_document_no,fiscal_year,line_item_no,account_type,debit_credit_indicator,amount_local_currency,amount_document_currency,assignment_no,...,payment_terms,cost_center_id,gl_account_no,customer_id,vendor_id,payement_baseline_date,tax_code,clearing_document_number,clearing_date,special_gl_indicator
0,100,C003,5105600151,2022,2,S,S,300.0,300.0,450000003500010,...,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN
1,50,C003,5105600143,2022,2,S,S,600.0,600.0,450000003400010,...,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN
2,100,C003,5105600152,2022,2,S,S,550.0,550.0,450000003600010,...,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN
3,100,C003,5105600153,2022,2,S,S,360.0,360.0,450000003900010,...,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN
4,100,C003,5105600154,2022,2,S,S,480.0,480.0,450000004000010,...,NaN,NaN,500010,NaN,NaN,NaN,None,NaN,NaN,NaN


,pandas_dtype
client_id,str
company_code,str
accounting_document_no,str
fiscal_year,int64
line_item_no,str
account_type,str
debit_credit_indicator,str
amount_local_currency,float64
amount_document_currency,float64
assignment_no,str


## 5. Missing values and blank strings

In [51]:
missing_profile = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values(["missing_percentage", "missing_count"], ascending=False)
display(missing_profile)

text_columns = df.select_dtypes(include=["object", "string"]).columns
blank_profile = (
    df[text_columns]
    .apply(lambda col: col.astype("string").str.strip().eq("").sum())
    .rename("blank_string_count")
    .to_frame()
)
display(blank_profile)

,missing_count,missing_percentage
tax_code,332106,100.00
special_gl_indicator,332101,100.00
line_item_text,332054,99.98
cost_center_id,332026,99.98
clearing_document_number,328652,98.96
clearing_date,328652,98.96
customer_id,293496,88.37
payment_terms,258029,77.69
payement_baseline_date,257694,77.59
vendor_id,144799,43.60


,blank_string_count
client_id,0
company_code,0
accounting_document_no,0
line_item_no,0
account_type,0
debit_credit_indicator,0
assignment_no,0
line_item_text,0
payment_terms,0
cost_center_id,0


## 6. Exact duplicate analysis

In [52]:
exact_duplicate_count = int(df.duplicated().sum())
duplicate_group_rows = int(df.duplicated(keep=False).sum())

print(f"Exact duplicate rows after first copies: {exact_duplicate_count:,}")
print(f"Rows belonging to exact-duplicate groups: {duplicate_group_rows:,}")
print(f"Exact duplicate percentage: {exact_duplicate_count / len(df) * 100:.2f}%" if len(df) else "Table is empty")

if exact_duplicate_count:
    display(df[df.duplicated(keep=False)].head(20))

Exact duplicate rows after first copies: 77
Rows belonging to exact-duplicate groups: 154
Exact duplicate percentage: 0.02%


,client_id,company_code,accounting_document_no,fiscal_year,line_item_no,account_type,debit_credit_indicator,amount_local_currency,amount_document_currency,assignment_no,...,payment_terms,cost_center_id,gl_account_no,customer_id,vendor_id,payement_baseline_date,tax_code,clearing_document_number,clearing_date,special_gl_indicator
188,50,C005,90000015,2022,1,D,S,18.38,18.38,0090000166,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
189,50,C005,90000017,2022,1,D,S,8.93,8.93,0090000168,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
190,50,C005,90000019,2022,1,D,S,115.50,115.50,0090000170,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
191,50,C005,90000000,2022,1,D,S,18.38,18.38,0090000151,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
192,50,C005,90000001,2022,1,D,S,8.93,8.93,0090000152,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
194,50,C005,90000003,2022,1,D,S,20.48,20.48,0090000154,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
195,50,C005,90000005,2022,1,D,S,11.03,11.03,0090000156,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
196,50,C005,90000007,2022,1,D,S,136.50,136.50,0090000158,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
197,50,C005,90000009,2022,1,D,S,16.28,16.28,0090000160,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN
198,50,C005,90000010,2022,1,D,S,2.10,2.10,NaN,...,false,NaN,121000,false,NaN,2022-02-01,None,NaN,NaN,NaN


## 7. Numeric-column profiling

In [53]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
if numeric_columns:
    display(df[numeric_columns].describe().T)
else:
    print("No numeric columns detected by Pandas.")

,count,mean,std,min,25%,50%,75%,max
fiscal_year,332106.0,2.020277e+03,1.309897e+00,2018.00,2019.00,2020.0,2022.0,2.023000e+03
amount_local_currency,332103.0,1.482847e+07,6.467846e+08,0.03,3133.23,59710.0,1171391.0,9.897921e+10
amount_document_currency,332014.0,1.470177e+07,6.468443e+08,0.03,3125.00,59484.0,1181163.0,9.897921e+10


## 8. Date-column profiling (optional)

Review the table schema and add actual date column names to `DATE_COLUMNS`.

In [54]:
DATE_COLUMNS = []  # Example: ["document_date", "posting_date"]

for col in DATE_COLUMNS:
    if col not in df.columns:
        print(f"Skipping {col!r}: column not found.")
        continue

    parsed = pd.to_datetime(df[col], errors="coerce")
    invalid = df[col].notna() & parsed.isna()

    print(f"\n{col}")
    print("Missing values:", int(df[col].isna().sum()))
    print("Non-null values that failed parsing:", int(invalid.sum()))
    print("Earliest parsed date:", parsed.min())
    print("Latest parsed date:", parsed.max())

## 9. Optional candidate-key check

Do not assume every table's key. Fill `CANDIDATE_KEY` only after checking the data dictionary or SAP field definitions. An empty list skips the check.

In [55]:
CANDIDATE_KEY = []  # Fill only after confirming the business key

if not CANDIDATE_KEY:
    print("Candidate-key check skipped: define CANDIDATE_KEY after confirming the table's business key.")
else:
    missing_key_columns = [col for col in CANDIDATE_KEY if col not in df.columns]
    if missing_key_columns:
        print("Key check skipped; columns not found:", missing_key_columns)
    else:
        null_key_rows = int(df[CANDIDATE_KEY].isna().any(axis=1).sum())
        duplicated_key_rows = int(df.duplicated(subset=CANDIDATE_KEY, keep=False).sum())
        key_group_sizes = df.groupby(CANDIDATE_KEY, dropna=False).size()

        print("Candidate key:", CANDIDATE_KEY)
        print("Rows with at least one null key component:", f"{null_key_rows:,}")
        print("Rows belonging to repeated-key groups:", f"{duplicated_key_rows:,}")
        print("Repeated key values:", f"{int((key_group_sizes > 1).sum()):,}")
        print("Maximum rows per key:", int(key_group_sizes.max()) if len(key_group_sizes) else 0)

        if duplicated_key_rows:
            display(
                df[df.duplicated(subset=CANDIDATE_KEY, keep=False)]
                .sort_values(CANDIDATE_KEY)
                .head(30)
            )

Candidate-key check skipped: define CANDIDATE_KEY after confirming the table's business key.


## 10. Export a compact profiling summary

In [56]:
summary = pd.DataFrame([
    {"metric": "schema", "value": SCHEMA_NAME},
    {"metric": "table", "value": TABLE_NAME},
    {"metric": "row_count", "value": len(df)},
    {"metric": "column_count", "value": df.shape[1]},
    {"metric": "exact_duplicate_rows_excluding_first_copies", "value": int(df.duplicated().sum())},
    {"metric": "rows_belonging_to_exact_duplicate_groups", "value": int(df.duplicated(keep=False).sum())},
    {"metric": "columns_with_missing_values", "value": int(df.isna().any().sum())},
])

output_file = f"{TABLE_NAME}_profiling_summary.csv"
summary.to_csv(output_file, index=False)
print("Saved:", output_file)
display(summary)

Saved: stg_bseg_profiling_summary.csv


,metric,value
0,schema,public
1,table,stg_bseg
2,row_count,332106
3,column_count,21
4,exact_duplicate_rows_excluding_first_copies,77
5,rows_belonging_to_exact_duplicate_groups,154
6,columns_with_missing_values,13


## 11. Findings log

Complete after reviewing the outputs:

- **Table and purpose:**
- **Source row count / staging row count comparison:**
- **Missing-value findings:**
- **Blank-string findings:**
- **Exact duplicate findings:**
- **Candidate key and repeated-key findings:**
- **Date/numeric anomalies requiring business validation:**
- **Actions recommended:**
- **Fields that may be legitimately optional:**
- **Limitations / follow-up checks:**

**Rules:** do not automatically fill nulls with zero, delete duplicates, or modify staging tables. Verify source mappings and business semantics first. Document findings and create separate cleaned datasets only after the cleaning rule is justified.